# untell — free $0 RL training (Kaggle T4)

Trains untell's single-pass RL rewriter against the **free open-detector ensemble**, warm-started by DPO on the **free HC3 human corpus**. No paid detector keys. See `docs/free-training-runbook.md`.

## Before Run All
1. **Settings → Accelerator → GPU T4 x1** (or P100).
2. **Settings → Internet → On**.
3. (Optional but recommended) **Add-ons → Secrets → add `HF_TOKEN`** = a Hugging Face *write* token, so the adapter is pushed to your HF repo and survives a killed session. Set `HF_REPO` below to `your-username/untell-grpo`.
4. **Run → Run All.**

Honest scope: reaches the *open-detector* ceiling and transfers to held-out open detectors. It does **not** prove it beats GPTZero/Originality/Turnitin (those need paid APIs in the loop).

In [ ]:
# EDIT THIS: your Hugging Face repo id (needs the HF_TOKEN secret). Leave as-is to skip HF push.
HF_REPO = ""  # e.g. "yourname/untell-grpo"  |  "" = don't push (adapter stays on the session disk)
STEPS   = 150  # GRPO steps (~4h on T4). Resume next session for more.
print("HF_REPO =", HF_REPO or "(none)", "| STEPS =", STEPS)

In [ ]:
# 1) Setup: fresh-clone untell (always latest main) + install training deps
import os, subprocess, sys
os.chdir("/kaggle/working")          # a dir that always exists (guards against a deleted cwd)
REPO = "/kaggle/working/untell"
subprocess.run(["rm", "-rf", REPO])  # always start clean so re-runs pull the newest code
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ssamba1/untell.git", REPO], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,full,eval]"], check=True)
# Kaggle ships torchao 0.10, which peft hard-rejects (needs >0.16) during LoRA injection. We use
# bitsandbytes for 4-bit, not torchao, so remove it to avoid the ImportError at train start.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
print("OK: fresh clone at", os.getcwd(), "| torchao removed")

In [ ]:
# 2) Pull the HF token from Kaggle Secrets (if you added one) so pushes work
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = tok
    os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
    print("HF_TOKEN loaded from Kaggle secret.")
except Exception as e:
    print("No HF_TOKEN secret (", e, ") — training still runs; HF push disabled.")
    HF_REPO = ""

In [ ]:
# 3) Smoke test — MUST pass before spending GPU hours (tiny model, 2 steps)
subprocess.run([sys.executable, "-m", "training.rl_humanizer", "--smoke"], check=True)
print("\nSMOKE OK — pipeline works end to end.")

In [ ]:
# 4) DPO warm-start on the FREE HC3 human corpus (no key, ~20-30 min)
cmd = [sys.executable, "-m", "training.dpo_humanizer",
       "--model", "Qwen/Qwen2.5-3B-Instruct",
       "--use-human-corpus", "--n", "500", "--load-4bit",
       "--out", "out/dpo-humanizer"]
if HF_REPO:
    cmd += ["--hub-id", HF_REPO + "-dpo"]
subprocess.run(cmd, check=True)

In [ ]:
# 5) (optional) incremental HF push every 10 min so a killed session keeps the latest checkpoint
import threading, time
def _push(repo, folder, every=600):
    while True:
        time.sleep(every)
        subprocess.run(["huggingface-cli", "upload", repo, folder, "--repo-type", "model", "--quiet"])
if HF_REPO:
    threading.Thread(target=_push, args=(HF_REPO, "out/rl-humanizer"), daemon=True).start()
    print("background HF push armed ->", HF_REPO)
else:
    print("no HF_REPO — skipping background push")

In [ ]:
# 6) GRPO on the FREE ensemble, warm-started from the DPO adapter (~4h for 150 steps)
#    --dpo-init merges the DPO adapter into the base before GRPO wraps a fresh LoRA.
#    No UNTELL_SURROGATE_DIR => reward = free weighted open-detector ensemble
#    (MAGE + RoBERTa + HC3 + Fast-DetectGPT). RADAR is left OFF: RADAR-Vicuna-7B is
#    too big to run beside 4-bit training on a free 16GB T4 (OOM). MAGE carries the hard signal.
env = dict(os.environ)
cmd = [sys.executable, "-m", "training.rl_humanizer",
       "--model", "Qwen/Qwen2.5-3B-Instruct",
       "--dpo-init", "out/dpo-humanizer",
       "--tier", "full", "--steps", str(STEPS), "--k", "6",
       "--load-4bit", "--reward-sim-floor", "0.82",
       "--out", "out/rl-humanizer"]
if HF_REPO:
    cmd += ["--hub-id", HF_REPO]
subprocess.run(cmd, check=True, env=env)

## Done
The trained LoRA adapter is at `out/rl-humanizer/` (and pushed to `HF_REPO` if set).

**Hit the 9h wall before finishing?** Start a new session, Run All, but change the GRPO cell to add `--resume out/rl-humanizer/checkpoint-125` (use the newest checkpoint number).

**Use it locally (no key):**
```bash
export UNTELL_POLICY_DIR=your-username/untell-grpo   # or the local out/rl-humanizer dir
python -m untell.scripts.run --rewriter auto "Furthermore, this underscores a transformative paradigm."
```
`get_rewriter()` auto-selects the trained policy — single forward pass, no API, no loop.

**Measure for free:** `--rewriter base` (A/B vs untuned), `python -m untell.scripts.score --tier full "<text>"`, or `--browser zerogpt` for a real free web detector in the loop.